# 03b_Stage2_ML

**ST498 Capstone | BDO Forward-Looking Default Risk**

## What this notebook does

Stage A forecast 12 US macroeconomic variables out to 2030 Q4. Stage B asks what those
forecasts imply for credit card defaults, and this notebook answers that with machine
learning methods.

The target is the US credit card delinquency rate (FRED: DRCCLACBS): the share of credit
card balances 30 or more days past due. The approach is a satellite model: regress the
delinquency rate on lagged macroeconomic variables, then feed the Stage A projections
through the fitted model to obtain a delinquency path to 2030 Q4. That path is what feeds
an IFRS 9 Expected Credit Loss calculation, which requires banks to base provisions on
forward-looking macroeconomic information rather than historical averages alone.

Input is `Stage1_final_regressors_US_Q.csv` from `02c_Stage1_WinnerSelection`.

## How this fits with the other Stage B notebook

`03a_Stage2_TimeSeries` covers the time-series track (ARMA, ARIMAX, SARIMAX). Both
notebooks forecast the same target over the same window from the same 2020 Q4 origin, so
the two model families can be compared directly in `03c_Stage2_WinnerSelection`.

## Method in one paragraph

Regressors are chosen by a four-method consensus vote run on training data only. Nine
models are then fitted and compared over a single evaluation origin at 2020 Q4. OLS is
pre-registered as the primary model and the deliverable, decided before any results were
seen; machine learning models are challengers. If a challenger wins, that is reported,
but OLS remains the deliverable because IFRS 9 model governance requires coefficients
that can be explained to an auditor.

Full procedure and the reasoning behind each choice: `Stage_B_ML_Methodology.pdf`.
Interpretation of results belongs in report Sections 5.2 and 6.2, not in this notebook.

| Section | Content |
|---|---|
| 0 | Imports and configuration |
| 1 | Data loading and validation |
| 2 | Joint regressor selection (four-method consensus vote) |
| 3 | Feature matrix construction (primary + robustness runs R1–R3) |
| 4 | Model fitting (nine models, inner tune/validation split) |
| 5 | Evaluation (per-horizon metrics, skill scores, Diebold-Mariano test) |
| 6 | Winner selection, production forecast, export |


## Section 0 - Imports and Configuration

All imports and constants live here, so no later cell introduces a dependency or a
magic number mid-notebook.

**Time splits.** Models are fitted on 1991 Q1 to 2020 Q4 and evaluated on 2021 Q1 to
2025 Q4, which the models never see during fitting or tuning. `TUNE_END` and `VAL_START`
divide the training block further: hyperparameters are searched on the earlier part and
chosen on the later part, keeping the evaluation window clean. `CLEAN_START` marks where
the target stops being spline-imputed and becomes genuinely observed.

**CANDIDATES.** The 12 regressors the selection vote draws from, each at the lag where
its cross-correlation with the delinquency rate peaked in `01_EDA` Section 5. The `_L2`
suffix means the variable enters two quarters before the delinquency rate it helps
explain.

**UNVALIDATED_STAGE_A.** Six of the 12 did not beat naive persistence in their own
Stage A forecasts. Any that survive the vote carry more forecast uncertainty than the
rest, and this is reported rather than treated as disqualifying.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LassoCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.base import clone

import xgboost as xgb
from scipy import stats

SEED = 42

NAVY, TEAL, AMBER = '#1F3864', '#17A589', '#E67E22'
RED, GREEN, GREY  = '#C0392B', '#27AE60', '#7F8C8D'
BLUE, LBLUE       = '#2E75B6', '#AED6F1'
TEMPLATE = 'plotly_white'

GITHUB_RAW     = 'https://raw.githubusercontent.com/hogandan85/ST-498/refs/heads/main/Data%20Collection'
REGRESSOR_FILE = f'{GITHUB_RAW}/Stage1_final_regressors_US_Q.csv'
OUT_DIR        = Path.cwd().parent / 'Stage1_Outputs'

TARGET     = 'us_delinquency_rate'
TARGET_RAW = 'us_delinquency_rate_raw'

TRAIN_START, TRAIN_END  = '1991-03-31', '2020-12-31'
TUNE_END                = '2016-12-31'
VAL_START,   VAL_END    = '2017-03-31', '2020-12-31'
EVAL_START,  EVAL_END   = '2021-03-31', '2025-12-31'
CLEAN_START, CLEAN_END  = '2022-09-30', '2025-12-31'
FCST_START,  FCST_END   = '2026-03-31', '2030-12-31'
COVID_START, COVID_END  = '2020-03-31', '2022-06-30'

HORIZONS = [1, 4, 8, 12, 20]

# Each variable at its peak CCF lag from 01_EDA Section 5.
# Bond yield enters as first difference: the level failed the EDA stationarity tests.
CANDIDATES = [
    'us_gdp_yoy_growth_L2',
    'us_unemployment_L0',
    'us_cpi_L0',
    'us_consumer_confidence_L2',
    'us_bond_yield_10y_d1_L2',
    'us_credit_qoq_growth_L6',
    'us_sp500_log_ret_L4',
    'us_vix_log_ret_L0',
    'us_house_price_yoy_L3',
    'us_indprod_yoy_L3',
    'us_oil_yoy_L2',
    'us_reer_diff_L0',
]

# CPI L6 competes only in the lag-rich robustness run (R2), not the primary vote.
CANDIDATES_R2 = CANDIDATES + ['us_cpi_L6']

# Stage A winners that failed the DM test against naive persistence (02c).
UNVALIDATED_STAGE_A = [
    'us_consumer_confidence', 'us_credit_qoq_growth', 'us_indprod_yoy',
    'us_reer_diff', 'us_sp500_log_ret', 'us_vix_log_ret',
]

print(f'Configuration loaded | {len(CANDIDATES)} primary candidates | seed {SEED}')
print(f'Output directory: {OUT_DIR}')

Configuration loaded | 12 primary candidates | seed 42
Output directory: c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Stage1_Outputs


## Section 1 - Data Loading and Validation

Input is `Stage1_final_regressors_US_Q.csv` from `02c`: 164 quarterly rows, 1990 Q1 to
2030 Q4. Rows to 2025 Q4 are historical observations; 2026 Q1 onward are Stage A
projections. The load raises rather than warns if a column is missing or the row count is
unexpected, so a stale input file cannot reach the results.

**Two target columns.** `us_delinquency_rate` is spline-adjusted over 2020 Q1 to 2022 Q2,
where policy support suppressed observed defaults, and is the modelling target.
`us_delinquency_rate_raw` keeps the original series and is used only in robustness run R3.

**Calendar quarters vs usable quarters.** A quarter is usable only if the target and every
regressor are observed. Lagged variables lose their leading quarters, and REER has no data
before 1994 Q2, so requiring all 12 candidates costs 13 of the 120 training quarters and
the usable sample starts at 1994 Q2. Table 1.2 shows which variables bind. The usable
sample therefore depends on the feature set, and the report quotes these figures rather
than the calendar length.

In [2]:
df = pd.read_csv(REGRESSOR_FILE, index_col=0, parse_dates=True).sort_index()

required = CANDIDATES + [TARGET, TARGET_RAW, 'covid_dummy']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f'Input file missing required columns: {missing}')
if len(df) != 164:
    raise ValueError(f'Expected 164 rows from 02c, found {len(df)}')

print(f'Loaded {df.shape[0]} rows x {df.shape[1]} columns '
      f'({df.index.min().date()} to {df.index.max().date()})')
print(f'All {len(required)} required columns present.')
if 'delinquency_spline' in df.columns:
    print('NOTE: delinquency_spline still present - 02c drop not applied in this version.')

df_train = df.loc[TRAIN_START:TRAIN_END].copy()
df_tune  = df.loc[TRAIN_START:TUNE_END].copy()
df_val   = df.loc[VAL_START:VAL_END].copy()
df_eval  = df.loc[EVAL_START:EVAL_END].copy()
df_clean = df.loc[CLEAN_START:CLEAN_END].copy()
df_full  = df.loc[TRAIN_START:EVAL_END].copy()
df_fcst  = df.loc[FCST_START:FCST_END].copy()


def q(ts):
    return f'{ts.year} Q{ts.quarter}'

blocks = [('Training block', df_train), ('Tune split', df_tune), ('Val split', df_val),
          ('Evaluation', df_eval), ('Clean sub-window', df_clean), ('Full panel', df_full)]

print('\nTime splits (calendar quarters | usable with all 12 candidates):')
for name, b in blocks:
    n_cal = len(b)
    n_use = len(b[CANDIDATES + [TARGET]].dropna())
    print(f'  {name:<17}: {q(b.index.min())} to {q(b.index.max())}   {n_cal:>3} | {n_use:>3}')

print(f'  {"Forecast":<17}: {q(df_fcst.index.min())} to {q(df_fcst.index.max())}   '
      f'{len(df_fcst):>3} | {len(df_fcst[CANDIDATES].dropna()):>3}')

Loaded 164 rows x 42 columns (1990-03-31 to 2030-12-31)
All 15 required columns present.

Time splits (calendar quarters | usable with all 12 candidates):
  Training block   : 1991 Q1 to 2020 Q4   120 | 107
  Tune split       : 1991 Q1 to 2016 Q4   104 |  91
  Val split        : 2017 Q1 to 2020 Q4    16 |  16
  Evaluation       : 2021 Q1 to 2025 Q4    20 |  20
  Clean sub-window : 2022 Q3 to 2025 Q4    14 |  14
  Full panel       : 1991 Q1 to 2025 Q4   140 | 127
  Forecast         : 2026 Q1 to 2030 Q4    20 |  20


In [3]:
starts = pd.DataFrame([
    {'Variable': c,
     'First valid': df[c].first_valid_index().date(),
     'NaNs in training block': int(df_train[c].isna().sum())}
    for c in CANDIDATES + [TARGET]
]).sort_values('First valid', ascending=False)

print('Table 1.2 - First valid observation per candidate')
display(starts.set_index('Variable'))

binding = df_train[CANDIDATES + [TARGET]].isna().any(axis=1)
print(f'Training rows lost to listwise deletion: {binding.sum()} of {len(df_train)}')
print(f'First complete case: {df_train[~binding].index.min().date()}')

Table 1.2 - First valid observation per candidate


,First valid,NaNs in training block
Variable,,
us_reer_diff_L0,1994-06-30,13
us_house_price_yoy_L3,1991-12-31,3
us_indprod_yoy_L3,1991-12-31,3
us_credit_qoq_growth_L6,1991-09-30,2
us_oil_yoy_L2,1991-09-30,2
us_sp500_log_ret_L4,1991-03-31,0
us_delinquency_rate,1991-03-31,0
us_bond_yield_10y_d1_L2,1990-12-31,0
us_gdp_yoy_growth_L2,1990-09-30,0


Training rows lost to listwise deletion: 13 of 120
First complete case: 1994-06-30


In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_full.index, y=df_full[TARGET_RAW],
    mode='lines', name='Raw observed',
    line=dict(color=NAVY, width=2)))

fig.add_trace(go.Scatter(
    x=df_full.index, y=df_full[TARGET],
    mode='lines', name='Spline-adjusted (model target)',
    line=dict(color=AMBER, width=2, dash='dot')))

# COVID spline window: target is reconstructed, not observed
fig.add_vrect(
    x0=COVID_START, x1=COVID_END,
    fillcolor='rgba(192,57,43,0.10)', line_width=0,
    annotation_text='COVID spline window', annotation_position='top left',
    annotation_font=dict(size=9, color=RED))

# Clean sub-window: evaluation quarters scored against genuinely observed values
fig.add_vrect(
    x0=CLEAN_START, x1=CLEAN_END,
    fillcolor='rgba(46,117,182,0.08)', line_width=0,
    annotation_text='Clean sub-window', annotation_position='top right',
    annotation_font=dict(size=9, color=BLUE))

# Evaluation origin. Annotation added separately: add_vline's own annotation breaks on
# date-string axes in older plotly (it averages the x coords to place the label).
fig.add_vline(x=EVAL_START, line_dash='dash', line_color=GREY, line_width=1.2)
fig.add_annotation(
    x=EVAL_START, y=0.02, yref='paper', text='Evaluation origin',
    showarrow=False, font=dict(size=9, color=GREY),
    xanchor='left', xshift=4, bgcolor='rgba(255,255,255,0.7)')

train_mean = df_train[TARGET].mean()
fig.add_hline(y=train_mean, line_dash='dot', line_color=GREY, line_width=1,
              annotation_text=f'Training mean = {train_mean:.2f}%',
              annotation_position='bottom right',
              annotation_font=dict(size=9, color=GREY))

fig.update_layout(
    title=dict(
        text=('<b>Figure 1.1 - Target Variable: Raw vs Spline-Adjusted</b>'
              f'<br><span style="font-size:11.5px;color:{GREY}">'
              'Red = spline reconstruction window | Blue = clean evaluation sub-window | '
              'the 6 quarters between them are scored against imputed values</span>'),
        font=dict(size=16, color=NAVY), x=0.015, xanchor='left', y=0.96, yanchor='top'),
    xaxis_title='Quarter', yaxis_title='Delinquency Rate (%)',
    template=TEMPLATE, height=480,
    legend=dict(orientation='h', yanchor='top', y=1.15, xanchor='center', x=0.5),
    margin=dict(t=115, b=55, l=70, r=30))
fig.show()

## Section 2 - Joint Regressor Selection

The report's earlier approach kept a variable if its univariate cross-correlation with the
delinquency rate cleared a significance threshold. A variable can pass that screen and
still be redundant, or change sign, once correlated variables enter alongside it (Bellotti
and Crook 2013, Tables 3 and 4). Selection is therefore run inside a multivariate
framework instead.

No single method is reliable on its own: stepwise depends on removal order, Lasso picks
arbitrarily among correlated variables, and tree importances favour variables with more
split points. Four methods vote instead, and a variable is kept if at least two of them
nominate it.

**The four methods.** Backward stepwise by AIC removes the least informative variable
until AIC stops improving. Lasso ranks variables by coefficient magnitude at a
cross-validated penalty. Random Forest permutation importance measures the accuracy drop
when a variable is shuffled. Gradient boosting ranks by contribution to reducing training
error. Each nominates five of the twelve candidates.

**Rules fixed in advance.** Everything runs on the training block only, so the evaluation
window cannot influence which variables are chosen. `covid_dummy` is forced into every
model as a structural break control and never competes in the vote. CPI competes on the
same terms as everything else. If the vote returned fewer than four or more than eight
variables, a pre-specified tie-break would take the top six by vote count then mean rank.

The retained set defines Equation 5.11 in the report. It is a result, reported in Section
6, not a methodology pre-specification.

**Sensitivity checks.** Two cells test whether the outcome depends on arbitrary choices.
The first re-runs the vote without REER, which has no data before 1994 Q2 and truncates
the selection sample by 13 quarters. The second varies how many variables each method
nominates. Both are reported as sensitivity; neither revises the primary specification.

**Diagnostics.** Table 2.2 checks coefficient signs against economic priors, fitted both
on all twelve candidates and on the selected set, so a sign that is stable across the two
can be distinguished from one that moves. Table 2.3 reports variance inflation factors
against the thresholds in Bellotti and Crook (2012, Section 3.2): below 5 for OLS, below
10 for regularised and nonlinear models. Both tables are diagnostic and feed no selection
decision. The final cell traces the GDP coefficient across nested specifications to locate
where its sign changes.

In [5]:
train_sel = df_train[CANDIDATES + [TARGET, 'covid_dummy']].dropna()

X_sel = train_sel[CANDIDATES]
y_sel = train_sel[TARGET].values

# Scaled copy shared by the Lasso and tree methods. Built once here so the four
# selection cells below can run in any order.
scaler_sel = StandardScaler()
X_sel_s    = scaler_sel.fit_transform(X_sel)

TOP_N          = 5   # each method nominates this many
VOTE_THRESHOLD = 2   # retained if nominated by at least this many methods

print(f'Selection sample: {len(train_sel)} quarters '
      f'({q(train_sel.index.min())} to {q(train_sel.index.max())})')
print(f'{len(CANDIDATES)} candidates | each method nominates {TOP_N} | '
      f'retained at >= {VOTE_THRESHOLD} of 4 votes')

Selection sample: 107 quarters (1994 Q2 to 2020 Q4)
12 candidates | each method nominates 5 | retained at >= 2 of 4 votes


In [6]:
def backward_stepwise_aic(X_df, y, feature_cols, top_n):
    """Backward elimination by AIC. Starts with all candidates and drops the variable
    whose removal most improves AIC, stopping at top_n or when no removal helps."""
    remaining = list(feature_cols)
    while len(remaining) > top_n:
        full_aic = sm.OLS(y, sm.add_constant(X_df[remaining].values)).fit().aic
        best_aic, to_remove = full_aic, None
        for feat in remaining:
            others = [f for f in remaining if f != feat]
            aic = sm.OLS(y, sm.add_constant(X_df[others].values)).fit().aic
            if aic < best_aic:
                best_aic, to_remove = aic, feat
        if to_remove is None:
            break
        remaining.remove(to_remove)
    return remaining[:top_n]

top5_aic = backward_stepwise_aic(X_sel, y_sel, CANDIDATES, TOP_N)

print(f'Backward AIC top {len(top5_aic)}:')
for v in top5_aic:
    print(f'  {v}')

Backward AIC top 5:
  us_gdp_yoy_growth_L2
  us_unemployment_L0
  us_cpi_L0
  us_consumer_confidence_L2
  us_credit_qoq_growth_L6


In [7]:
lasso_cv = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000, random_state=SEED)
lasso_cv.fit(X_sel_s, y_sel)

lasso_ranked = sorted(
    [(c, abs(w)) for c, w in zip(CANDIDATES, lasso_cv.coef_) if abs(w) > 1e-6],
    key=lambda t: t[1], reverse=True)
top5_lasso = [c for c, _ in lasso_ranked[:TOP_N]]

print(f'Lasso (alpha={lasso_cv.alpha_:.4f}) nominated {len(lasso_ranked)} '
      f'non-zero, top {len(top5_lasso)} by |coefficient|:')
for c, w in lasso_ranked[:TOP_N]:
    print(f'  {c}: {w:.4f}')

Lasso (alpha=0.0153) nominated 12 non-zero, top 5 by |coefficient|:
  us_credit_qoq_growth_L6: 0.7711
  us_house_price_yoy_L3: 0.5517
  us_consumer_confidence_L2: 0.3635
  us_gdp_yoy_growth_L2: 0.2823
  us_unemployment_L0: 0.2449


In [8]:
rf_sel = RandomForestRegressor(n_estimators=200, max_depth=4,
                               random_state=SEED, n_jobs=-1)
rf_sel.fit(X_sel_s, y_sel)

perm    = permutation_importance(rf_sel, X_sel_s, y_sel, n_repeats=20,
                                 random_state=SEED, scoring='neg_mean_absolute_error')
rf_imp  = dict(zip(CANDIDATES, perm.importances_mean))
top5_rf = sorted(rf_imp, key=rf_imp.get, reverse=True)[:TOP_N]

print(f'Random Forest permutation importance top {TOP_N}:')
for v in top5_rf:
    print(f'  {v}: {rf_imp[v]:.4f}')

Random Forest permutation importance top 5:
  us_credit_qoq_growth_L6: 0.5887
  us_unemployment_L0: 0.1277
  us_house_price_yoy_L3: 0.1194
  us_indprod_yoy_L3: 0.0616
  us_cpi_L0: 0.0468


In [9]:
gb_sel = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                   learning_rate=0.05, random_state=SEED)
gb_sel.fit(X_sel_s, y_sel)

gb_imp  = dict(zip(CANDIDATES, gb_sel.feature_importances_))
top5_gb = sorted(gb_imp, key=gb_imp.get, reverse=True)[:TOP_N]

print(f'Gradient Boosting importance top {TOP_N}:')
for v in top5_gb:
    print(f'  {v}: {gb_imp[v]:.4f}')

Gradient Boosting importance top 5:
  us_credit_qoq_growth_L6: 0.4076
  us_house_price_yoy_L3: 0.1453
  us_unemployment_L0: 0.1306
  us_indprod_yoy_L3: 0.1032
  us_sp500_log_ret_L4: 0.0673


In [10]:
nominations = {'AIC': top5_aic, 'Lasso': top5_lasso, 'RF': top5_rf, 'GB': top5_gb}

# Rank within each method's top-N; variables not nominated get TOP_N + 1.
vote_counts, mean_ranks = {}, {}
for c in CANDIDATES:
    ranks = [(m.index(c) + 1) if c in m else TOP_N + 1 for m in nominations.values()]
    vote_counts[c] = sum(1 for m in nominations.values() if c in m)
    mean_ranks[c]  = np.mean(ranks)

vote_df = pd.DataFrame({
    'Variable':  CANDIDATES,
    'Votes':     [vote_counts[c] for c in CANDIDATES],
    'Mean rank': [round(mean_ranks[c], 2) for c in CANDIDATES],
    **{m: ['Yes' if c in lst else 'No' for c in CANDIDATES]
       for m, lst in nominations.items()},
}).sort_values(['Votes', 'Mean rank'], ascending=[False, True]).reset_index(drop=True)

print('Table 2.1 - Four-method consensus vote')
display(vote_df.set_index('Variable'))

consensus = vote_df.loc[vote_df['Votes'] >= VOTE_THRESHOLD, 'Variable'].tolist()

# Pre-specified tie-break: if the vote returns fewer than 4 or more than 8, fall back to
# the top 6 by vote count then mean rank. Recorded whether or not it fires.
if len(consensus) < 4 or len(consensus) > 8:
    consensus = vote_df['Variable'].head(6).tolist()
    print('\nTie-break applied: vote returned an out-of-range set, top 6 taken.')
else:
    print(f'\nTie-break not triggered ({len(consensus)} variables, within 4-8).')

FEATURES_EQ511         = consensus + ['covid_dummy']
FEATURES_EQ511_NODUMMY = consensus

unval = [c for c in consensus if any(c.startswith(u) for u in UNVALIDATED_STAGE_A)]

print(f'\nRetained ({len(consensus)} of {len(CANDIDATES)}):')
for c in consensus:
    print(f'  {c}{"  [Stage A unvalidated]" if c in unval else ""}')
print(f'\nRejected: {[c for c in CANDIDATES if c not in consensus]}')
print(f'\nThis set defines Equation 5.11. The equation is a result, reported in '
      f'Section 6, not a methodology pre-specification.')
print(f'{len(unval)} of {len(consensus)} retained variables lack Stage A DM validation.')

Table 2.1 - Four-method consensus vote


,Votes,Mean rank,AIC,Lasso,RF,GB
Variable,,,,,,
us_credit_qoq_growth_L6,4,2.00,Yes,Yes,Yes,Yes
us_unemployment_L0,4,3.00,Yes,Yes,Yes,Yes
us_house_price_yoy_L3,3,3.25,No,Yes,Yes,Yes
us_gdp_yoy_growth_L2,2,4.25,Yes,Yes,No,No
us_consumer_confidence_L2,2,4.75,Yes,Yes,No,No
us_cpi_L0,2,5.00,Yes,No,Yes,No
us_indprod_yoy_L3,2,5.00,No,No,Yes,Yes
us_sp500_log_ret_L4,1,5.75,No,No,No,Yes
us_bond_yield_10y_d1_L2,0,6.00,No,No,No,No



Tie-break not triggered (7 variables, within 4-8).

Retained (7 of 12):
  us_credit_qoq_growth_L6  [Stage A unvalidated]
  us_unemployment_L0
  us_house_price_yoy_L3
  us_gdp_yoy_growth_L2
  us_consumer_confidence_L2  [Stage A unvalidated]
  us_cpi_L0
  us_indprod_yoy_L3  [Stage A unvalidated]

Rejected: ['us_bond_yield_10y_d1_L2', 'us_sp500_log_ret_L4', 'us_vix_log_ret_L0', 'us_oil_yoy_L2', 'us_reer_diff_L0']

This set defines Equation 5.11. The equation is a result, reported in Section 6, not a methodology pre-specification.
3 of 7 retained variables lack Stage A DM validation.


In [12]:
def run_vote(candidates, df_block, top_n=TOP_N, threshold=VOTE_THRESHOLD):
    """Full four-method vote on an arbitrary candidate list. Used for sensitivity checks;
    the primary vote above is run cell by cell so each method's output is visible."""
    s  = df_block[candidates + [TARGET]].dropna()
    Xd = s[candidates]
    y  = s[TARGET].values
    Xs = StandardScaler().fit_transform(Xd)

    a = backward_stepwise_aic(Xd, y, candidates, top_n)

    lc = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000,
                 random_state=SEED).fit(Xs, y)
    l  = [c for c, _ in sorted([(c, abs(w)) for c, w in zip(candidates, lc.coef_)
                                if abs(w) > 1e-6],
                               key=lambda t: t[1], reverse=True)[:top_n]]

    rf = RandomForestRegressor(n_estimators=200, max_depth=4,
                               random_state=SEED, n_jobs=-1).fit(Xs, y)
    pi = permutation_importance(rf, Xs, y, n_repeats=20, random_state=SEED,
                                scoring='neg_mean_absolute_error')
    r  = [candidates[i] for i in np.argsort(pi.importances_mean)[::-1][:top_n]]

    gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                                   random_state=SEED).fit(Xs, y)
    g  = [candidates[i] for i in np.argsort(gb.feature_importances_)[::-1][:top_n]]

    counts = {c: sum(c in lst for lst in [a, l, r, g]) for c in candidates}
    return sorted([c for c in candidates if counts[c] >= threshold],
                  key=lambda c: -counts[c]), len(s)

# Guard: run_vote reimplements the four methods, so confirm it reproduces the explicit
# vote above before any sensitivity result is trusted.
_check, _ = run_vote(CANDIDATES, df_train)
assert set(_check) == set(consensus), 'run_vote diverges from the explicit primary vote'

# REER truncates the selection sample by 13 quarters (no data before 1994 Q2). The
# justification for re-running without it is data availability, which is knowable without
# reference to any vote outcome, so this is not result-dependent selection.
cand_no_reer   = [c for c in CANDIDATES if not c.startswith('us_reer')]
sens_set, sens_n = run_vote(cand_no_reer, df_train)

print(f'Primary vote : {len(train_sel)} quarters, {len(consensus)} retained')
print(f'Without REER : {sens_n} quarters, {len(sens_set)} retained')
print(f'\nSensitivity set: {sens_set}')

added   = [c for c in sens_set if c not in consensus]
dropped = [c for c in consensus if c not in sens_set]
if not added and not dropped:
    print('\nIdentical to the primary vote. REER truncation does not affect selection.')
else:
    print(f'\nDiffers from primary vote. Added: {added} | Dropped: {dropped}')
    print('This difference must be reported in Section 6.')

Primary vote : 107 quarters, 7 retained
Without REER : 117 quarters, 7 retained

Sensitivity set: ['us_gdp_yoy_growth_L2', 'us_credit_qoq_growth_L6', 'us_unemployment_L0', 'us_house_price_yoy_L3', 'us_cpi_L0', 'us_consumer_confidence_L2', 'us_indprod_yoy_L3']

Identical to the primary vote. REER truncation does not affect selection.


In [13]:
# TOP_N = 5 is the pre-registered value. This records how the retained set would change
# under alternatives. Reported as sensitivity; it does not revise the primary set.
print(f'{"top_n":>6} {"Kept":>5}  Variables')
for tn in [4, 5, 6, 7]:
    s, _ = run_vote(CANDIDATES, df_train, top_n=tn)
    mark = '  <- pre-registered' if tn == TOP_N else ''
    print(f'{tn:>6} {len(s):>5}  {[c.replace("us_", "") for c in s]}{mark}')

 top_n  Kept  Variables
     4     6  ['unemployment_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'gdp_yoy_growth_L2', 'consumer_confidence_L2', 'indprod_yoy_L3']
     5     7  ['unemployment_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'gdp_yoy_growth_L2', 'cpi_L0', 'consumer_confidence_L2', 'indprod_yoy_L3']  <- pre-registered
     6     8  ['unemployment_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'cpi_L0', 'gdp_yoy_growth_L2', 'consumer_confidence_L2', 'sp500_log_ret_L4', 'indprod_yoy_L3']
     7     8  ['gdp_yoy_growth_L2', 'unemployment_L0', 'cpi_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'indprod_yoy_L3', 'consumer_confidence_L2', 'sp500_log_ret_L4']


In [14]:
# Economic priors set before fitting. Ambiguous cases are recorded as such rather than
# assigned a direction, so the check cannot be passed by hindsight.
SIGN_PRIORS = {
    'us_gdp_yoy_growth_L2':      ('-', 'growth reduces defaults'),
    'us_unemployment_L0':        ('+', 'joblessness raises defaults'),
    'us_cpi_L0':                 ('?', 'erodes real debt burden but also real income'),
    'us_consumer_confidence_L2': ('-', 'confidence reduces defaults'),
    'us_bond_yield_10y_d1_L2':   ('+', 'rising rates raise debt service'),
    'us_credit_qoq_growth_L6':   ('+', 'credit expansion precedes over-leverage'),
    'us_sp500_log_ret_L4':       ('-', 'equity gains support household balance sheets'),
    'us_vix_log_ret_L0':         ('+', 'volatility signals stress'),
    'us_house_price_yoy_L3':     ('-', 'housing wealth and collateral support repayment'),
    'us_indprod_yoy_L3':         ('-', 'activity reduces defaults'),
    'us_oil_yoy_L2':             ('+', 'energy costs squeeze disposable income'),
    'us_reer_diff_L0':           ('?', 'competitiveness channel is indirect'),
}

def fit_hac(features):
    s = df_train[features + [TARGET]].dropna()
    X = sm.add_constant(s[features].values, has_constant='add')
    return sm.OLS(s[TARGET].values, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

ols_all12 = fit_hac(CANDIDATES + ['covid_dummy'])
ols_sel   = fit_hac(FEATURES_EQ511)
sel_coefs = dict(zip(FEATURES_EQ511, ols_sel.params[1:]))

rows = []
for i, c in enumerate(CANDIDATES):
    b12   = ols_all12.params[i + 1]
    prior = SIGN_PRIORS[c][0]
    got   = '+' if b12 > 0 else '-'
    b7    = sel_coefs.get(c, np.nan)
    rows.append({
        'Variable':      c,
        'Prior':         prior,
        'All 12':        round(b12, 4),
        'p':             round(ols_all12.pvalues[i + 1], 4),
        'Selected 7':    round(b7, 4) if c in sel_coefs else '-',
        'Matches prior': 'n/a' if prior == '?' else ('Yes' if got == prior else 'NO'),
        'Sign stable':   ('-' if c not in sel_coefs
                          else 'Yes' if np.sign(b12) == np.sign(b7) else 'FLIP'),
    })

print('Table 2.2 - Joint sign diagnostic (Newey-West HAC, maxlags=4). Diagnostic only.')
display(pd.DataFrame(rows).set_index('Variable'))

viol = [r['Variable'] for r in rows if r['Matches prior'] == 'NO']
flip = [r['Variable'] for r in rows if r['Sign stable'] == 'FLIP']
print(f'Prior violations: {viol if viol else "none"}')
print(f'Sign flips between the 12-variable and 7-variable fits: {flip if flip else "none"}')

Table 2.2 - Joint sign diagnostic (Newey-West HAC, maxlags=4). Diagnostic only.


,Prior,All 12,p,Selected 7,Matches prior,Sign stable
Variable,,,,,,
us_gdp_yoy_growth_L2,-,0.1769,0.0133,0.2039,NO,Yes
us_unemployment_L0,+,0.2499,0.0008,0.2586,Yes,Yes
us_cpi_L0,?,0.1104,0.1116,0.2075,n/a,Yes
us_consumer_confidence_L2,-,-0.3492,0.0000,-0.3376,Yes,Yes
us_bond_yield_10y_d1_L2,+,0.0401,0.6957,-,Yes,-
us_credit_qoq_growth_L6,+,0.9238,0.0000,0.8799,Yes,Yes
us_sp500_log_ret_L4,-,-0.0356,0.9734,-,Yes,-
us_vix_log_ret_L0,+,0.4638,0.0221,-,Yes,-
us_house_price_yoy_L3,-,-0.0956,0.0000,-0.1008,Yes,Yes


Prior violations: ['us_gdp_yoy_growth_L2']
Sign flips between the 12-variable and 7-variable fits: none


In [15]:
vif_data = df_train[FEATURES_EQ511].dropna()
vif_mat  = sm.add_constant(vif_data.values, has_constant='add')

vif_df = pd.DataFrame({
    'Variable': list(vif_data.columns),
    'VIF': [round(variance_inflation_factor(vif_mat, i + 1), 2)
            for i in range(vif_data.shape[1])],
}).sort_values('VIF', ascending=False).reset_index(drop=True)

# Thresholds from Bellotti and Crook (2012), Section 3.2: below 5 for OLS, below 10 for
# regularised and nonlinear models, which tolerate correlated inputs.
vif_df['Assessment'] = ['OK for OLS' if v < 5 else
                        'OK for regularised only' if v < 10 else 'High'
                        for v in vif_df['VIF']]

print(f'Table 2.3 - Variance Inflation Factors (n={len(vif_data)})')
display(vif_df.set_index('Variable'))

high = vif_df.loc[vif_df['VIF'] >= 5, 'Variable'].tolist()
print(f'Above 5 (OLS threshold): {high if high else "none"}')

Table 2.3 - Variance Inflation Factors (n=117)


,VIF,Assessment
Variable,,
us_gdp_yoy_growth_L2,3.65,OK for OLS
us_unemployment_L0,3.63,OK for OLS
us_indprod_yoy_L3,3.04,OK for OLS
us_house_price_yoy_L3,2.51,OK for OLS
us_consumer_confidence_L2,2.41,OK for OLS
us_credit_qoq_growth_L6,1.65,OK for OLS
us_cpi_L0,1.28,OK for OLS
covid_dummy,1.26,OK for OLS


Above 5 (OLS threshold): none


In [16]:
# GDP enters positive against a negative prior, significantly. This traces where the
# reversal happens: alone, then conditioned on the other activity measures.
gdp, unemp, indprod = 'us_gdp_yoy_growth_L2', 'us_unemployment_L0', 'us_indprod_yoy_L3'

for label, feats in [('GDP alone', [gdp]),
                     ('GDP + unemployment', [gdp, unemp]),
                     ('GDP + unemp + indprod', [gdp, unemp, indprod]),
                     ('Full selected set', FEATURES_EQ511)]:
    sub = df_train[feats + [TARGET]].dropna()
    m = sm.OLS(sub[TARGET].values,
               sm.add_constant(sub[feats].values, has_constant='add')
               ).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    k = feats.index(gdp) + 1          # +1 for the intercept
    print(f'{label:<24} GDP coef = {m.params[k]:+.4f}  '
          f'(p={m.pvalues[k]:.4f}, n={len(sub)})')

s       = df_train[[gdp, TARGET]].dropna()
vif_gdp = vif_df.loc[vif_df['Variable'] == gdp, 'VIF'].iloc[0]
print(f'\nPearson correlation, GDP vs delinquency: {s[gdp].corr(s[TARGET]):+.4f}')
print(f'GDP variance explained by other selected regressors: {1 - 1/vif_gdp:.1%}')

GDP alone                GDP coef = -0.1113  (p=0.3943, n=120)
GDP + unemployment       GDP coef = -0.0422  (p=0.6986, n=120)
GDP + unemp + indprod    GDP coef = +0.0577  (p=0.5830, n=117)
Full selected set        GDP coef = +0.2039  (p=0.0074, n=117)

Pearson correlation, GDP vs delinquency: -0.1842
GDP variance explained by other selected regressors: 72.6%
